# PHA 领域专家 Agent 示例

本 Notebook 展示 PHA 的**领域专家 Agent**。它通过 OneTwo 使用 **ReAct 框架**，逐步推理健康问题，借助网页搜索、Data Commons、代码执行和参考范围等工具解读结果。

## 环境要求

安装 OneTwo 可启用完整的 ReAct 功能。

```bash
pip install git+https://github.com/google-deepmind/onetwo
```

缺少 OneTwo 时，Agent 会以简化模式运行。

## 环境搭建

In [ ]:
# 安装依赖关系( 需要时不做评论)
# !pip install git+https://github.com/google-deepmind/onetwo  # For ReAct support
# !pip install google-genai openai # LLM backends
# !pip install tavily-python  # Better search (optional)
# !pip install duckduckgo-search  # Free search fallback

In [ ]:
import os
import sys

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"

# 可选：Tavily API 键 领域专家 Agent (搜索).
TAVILY_API_KEY = "your-tavily-api-key-here"


In [ ]:
# 检查是否有1 2 个可用
from pha.agents import is_react_available

print(f"OneTwo ReAct 可用： {is_react_available()}")
if not is_react_available():
    print("当前使用简化模式，完整 ReAct 功能不可用。")
    print("如需启用完整功能，请运行： pip install git+https://github.com/google-deepmind/onetwo")

## 初始化领域专家 Agent

In [ ]:
from pha.agents import DomainExpertAgent

# 创建代理
# 默认使用Tavily进行高质量搜索(需要TAVILY_API_KEY).
# 设定 search_backend ='duckduckgo' 用于免费但质量较低的搜索
agent = DomainExpertAgent(
    search_backend='tavily',  # 或"Duckduckgo"如果你没有 Tavily 键
    tavily_api_key=TAVILY_API_KEY,
)

# 配置设置( 使用全局 LLM 配置)
import glob

# 查找实例笔记本( PHIA 模式)
exemplar_files = glob.glob('../few_shots/*.ipynb')
print(f"找到 {len(exemplar_files)} 本少样本 Notebook")

agent.get_agent(
    api_key=API_KEY,
    provider=PROVIDER,
    exemplar_files=exemplar_files,
)

print("领域专家 Agent 已初始化")
print(f"正在使用 ReAct： {is_react_available()}")


## 提供用户健康数据（可选）

Agent 可以结合用户健康数据给出个性化回答。

In [ ]:
# 手动设定用户健康数据
agent.set_user_health_data("""
## 用户档案
- 年龄：45 岁
- 生理性别：男
- 身高：178 cm
- 体重：82 kg
- BMI: 25.9

## 最近的化验结果
- 低密度脂蛋白胆固醇（LDL-C）： 142 mg/dL
- 高密度脂蛋白胆固醇（HDL-C）： 52 mg/dL
- 总胆固醇： 210 mg/dL
- 甘油三酯： 145 mg/dL
- 空腹血糖： 98 mg/dL
- HbA1c: 5.7%

## 可穿戴设备数据（7 天平均）
- 每日步数： 7,500
- 静息心率： 68 bpm
- 心率变异性： 42 ms
- 睡眠时长：6.5 小时
""")

print("已设置用户健康数据")

## 提出健康问题

Agent 会按 ReAct 流程处理。

1. 分析问题。
2. 调用工具，如搜索、计算或查询参考范围。
3. 查看返回结果。
4. 循环执行，直到得到回答。

In [ ]:
# 示例1:解释实验室结果。
question = "从化验结果看，我需要留意胆固醇水平吗？"

print(f"问题：{question}")
print("\nAgent 正在处理…")

response = agent.call_agent(question)
print(f"\n回复：\n{response}")

In [ ]:
# 示例 2:询问具体衡量标准
question = "糖化血红蛋白（HbA1c）为 5.7%，这对我的糖尿病风险意味着什么？"

print(f"问题：{question}")
print("\nAgent 正在处理…")

response = agent.call_agent(question)
print(f"\n回复：\n{response}")

In [ ]:
# 示例3:询问可穿戴的度量衡
question = "与我这个年龄段的健康参考值相比，我的静息心率和心率变异性如何？"

print(f"问题：{question}")
print("\nAgent 正在处理…")

response = agent.call_agent(question)
print(f"\n回复：\n{response}")

In [ ]:
# 实例4:全面健康评估
question = """结合我的健康数据，有哪三件事最值得优先做？请给出具体、可执行的建议。"""

print(f"问题：{question}")
print("\nAgent 正在处理…")

response = agent.call_agent(question)
print(f"\n回复：\n{response}")

## 从样本文件加载健康数据

In [ ]:
# 装入样本数据
from config import Settings
from pha.utils import load_persona

settings = Settings(data_dir='../data/sample')
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

# 格式化代理商的最近数据
recent_7_days = summary_df.tail(7)

health_summary = f"""
## 用户档案
- Age: {profile_df['age'].iloc[0]}
- Sex: {profile_df['sex'].iloc[0]}
- Height: {profile_df['height_cm'].iloc[0]} cm
- Weight: {profile_df['weight_kg'].iloc[0]} kg

## 7-Day Wearable Summary (averages)
- 每日步数： {recent_7_days['steps'].mean():.0f}
- Sleep Duration: {recent_7_days['sleep_minutes'].mean():.0f} minutes
- 静息心率： {recent_7_days['resting_heart_rate'].mean():.1f} bpm
- 心率变异性： {recent_7_days['heart_rate_variability'].mean():.1f} ms
- Sleep Score: {recent_7_days['sleep_score'].mean():.1f}
"""

agent.set_user_health_data(health_summary)
print("已加载健康数据样本：")
print(health_summary)

In [ ]:
# 询问装入的数据
question = "结合我的可穿戴设备数据，我的睡眠质量如何？怎样改善？"

print(f"问题：{question}")
print("\nAgent 正在处理…")

response = agent.call_agent(question)
print(f"\n回复：\n{response}")

## 自己试着提问

In [ ]:
# 问你自己的健康问题
your_question = "调整哪些生活习惯最有助于改善我的心血管健康？"

response = agent.call_agent(your_question)
print(response)